# Phase 4: does AST chunking improve retrieval?

This notebook runs the project's real chunkers, Qwen3 embeddings, Chroma, BM25 and weighted
RRF on **40 Requests and 40 Express questions**, at pinned repository revisions. It includes
development and held-out test splits. It does not need an LLM or a GitHub token.

1. Upload **phase4_source.zip** to the top level of **My Drive** in Google Drive.
2. Open this notebook in Colab. Runtime → Change runtime type → select a **GPU**.
   The first full run is verified on an A100 40 GB; other GPU types have not been measured.
3. Runtime → Run all, and approve Colab's request to mount your Drive. The notebook loads
   the ZIP from Drive, verifies it matches this notebook, and extracts the project locally.
4. Keep the session open until `phase4_results.zip` downloads, then bring it back to the chat.

If you put the source ZIP in another Drive folder, change `SOURCE_ZIP` in the Parameters cell.
Download the notebook and source ZIP from the same generation. The notebook contains no
embedded project archive or long encoded payload.

The primary comparison runs four chunkers, with and without identical contextual headers.
The reference `astchunk` does not support JavaScript: Express reports that row as N/A.
The optional final experiment reranks the current AST/header-on candidates at depths 20 and
50, using a code-specific instruction. It downloads the 4B reranker and takes extra GPU time.

No scores are known in advance. A result favouring character chunks is a valid outcome.
Results include overlap, union-of-gold-line coverage and ≥90%-complete gold ranges, never our
AST chunker's symbol metadata. Reports include all 40 questions for fixed comparisons plus
separate dev/test tables and paired evidence-group bootstrap intervals. Selected settings are
assessed on test only; pooled summaries are descriptive. Intervals are pointwise/exploratory,
and the character versus non-whitespace budget confound still applies.

## Parameters

In [ ]:
SOURCE_ZIP = "/content/drive/MyDrive/phase4_source.zip"
RUN_RERANK = True  # Set False to skip the 4B reranker experiment on both repos.
DTYPE = "float32"  # Match CPU laptop embeddings.

## Load project ZIP from Google Drive

In [ ]:
# Load the separate source ZIP that you uploaded to Google Drive.
import hashlib, pathlib, zipfile
from google.colab import drive

drive.mount("/content/drive")
source_zip = pathlib.Path(SOURCE_ZIP)
if not source_zip.is_file():
    raise FileNotFoundError(
        f"Upload phase4_source.zip to My Drive, or update SOURCE_ZIP. Missing: {source_zip}"
    )
expected_sha256 = "29a33dbb2a68affaf2658cb2370fcf043bb172531673384891f55f0c18cedf31"
if hashlib.sha256(source_zip.read_bytes()).hexdigest() != expected_sha256:
    raise ValueError(
        "Source ZIP does not match this notebook. Download the matching ZIP and notebook again."
    )
PAYLOAD = pathlib.Path("/content/phase4_payload")
with zipfile.ZipFile(source_zip) as package:
    package.extractall(PAYLOAD)
print("Loaded project code and question datasets from", source_zip.name)

## Install dependencies with uv

In [ ]:
import importlib.util, subprocess, sys
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], check=True)
if importlib.util.find_spec("uv") is None:
    # Bootstrap the installer once if this Colab runtime does not include it.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
# uv installs the pinned dependencies into the same Python environment as this notebook.
subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--system",
                "--python", sys.executable, "-q", 'chromadb==1.5.9', 'fastapi==0.162.0', 'gitpython==3.2.1', 'langchain-chroma==1.1.0', 'langchain-core==1.6.13', 'langchain-text-splitters==1.1.3', 'langgraph==1.2.19', 'pydantic==2.14.1', 'pydantic-settings==2.16.1', 'rank-bm25==0.2.2', 'rich==15.0.0', 'sentence-transformers==5.7.0', 'transformers==4.57.6', 'tree-sitter==0.26.0', 'tree-sitter-language-pack==1.22.1', 'typer==0.27.3', 'uvicorn==0.54.0', 'astchunk==0.1.0'], check=True)

## Validate and compare

In [ ]:
import json, os, subprocess, sys, time

RESULTS = pathlib.Path("/content/phase4_results")
REPOS = pathlib.Path("/content/phase4_repos")
REPOS.mkdir(exist_ok=True)
ENV = {
    **os.environ,
    "PYTHONPATH": os.pathsep.join([str(PAYLOAD), str(PAYLOAD / "src")]),
    "REPO_RAG_DATA_DIR": "/content/phase4_unused_production_data",
    "REPO_RAG_COMPUTE_DEVICE": "cuda",
    "REPO_RAG_EMBEDDING_DTYPE": DTYPE,
    "PYTHONUNBUFFERED": "1",
    "TOKENIZERS_PARALLELISM": "false",
}

def run_eval(repo, checkout, *extra, output=RESULTS):
    subprocess.run([
        sys.executable, str(PAYLOAD / "eval/run_retrieval_eval.py"),
        "--repo", repo, "--checkout", str(checkout), "--output", str(output), *extra,
    ], env=ENV, check=True)

checkouts = {}
for dataset in sorted((PAYLOAD / "eval/datasets").glob("*_qa.jsonl")):
    first = json.loads(dataset.read_text().splitlines()[0])
    repo, sha = first["repo"], first["commit_sha"]
    checkout = REPOS / repo.replace("/", "__")
    if not checkout.exists():
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                        "https://github.com/" + repo, str(checkout)], check=True)
    subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", sha], check=True)
    checkouts[repo] = checkout
    run_eval(repo, checkout, "--mode", "validate")

started = time.perf_counter()
for repo, checkout in checkouts.items():
    print("\nFull comparison:", repo, flush=True)
    run_eval(repo, checkout, "--mode", "full")
print(f"Main comparison elapsed: {time.perf_counter() - started:.1f} seconds")

## Optional reranker experiment

In [ ]:
if RUN_RERANK:
    started = time.perf_counter()
    for repo, checkout in checkouts.items():
        print("\nReranker depth experiment:", repo, flush=True)
        run_eval(repo, checkout, "--mode", "full", "--strategies", "ast", "--headers", "on",
                 "--rerank", output=RESULTS / "rerank_experiment")
    print(f"Reranker experiment elapsed: {time.perf_counter() - started:.1f} seconds")
else:
    print("Reranking skipped; its code-search effect remains unmeasured.")

## Download results

In [ ]:
# Export human-readable reports, per-question rankings and portable vector caches.
# Exclude Chroma's platform-specific binary index.
from google.colab import files
archive_path = pathlib.Path("/content/phase4_results.zip")
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RESULTS.rglob("*")):
        if path.is_file() and "isolated_index" not in path.parts:
            bundle.write(path, path.relative_to(RESULTS))
print(f"Results archive: {archive_path.stat().st_size / 1e6:.1f} MB")
for path in sorted(RESULTS.rglob("summary.md")):
    print(path.relative_to(RESULTS))
files.download(str(archive_path))